# TP — Régression Linéaire sur le Dataset Car Details
## Méthodologie : CRISP-DM

> **Dataset :** [Car Details from Car Dekho — Kaggle](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho)
> **Objectif :** Appliquer la régression linéaire simple et multiple sur des données réelles en suivant les 6 phases de CRISP-DM.

---

## Rappel : Les 6 phases de CRISP-DM

| Phase | Nom | Ce qu'on fait |
|-------|-----|---------------|
| 1 | **Business Understanding** | Définir le problème métier |
| 2 | **Data Understanding** | Explorer et comprendre les données |
| 3 | **Data Preparation** | Nettoyer et transformer les données |
| 4 | **Modeling** | Construire le(s) modèle(s) de régression |
| 5 | **Evaluation** | Mesurer la qualité du modèle et vérifier les hypothèses |
| 6 | **Deployment** | Interpréter et communiquer les résultats |

---

---
## Phase 0 — Installation et Imports

Importez toutes les librairies nécessaires pour ce TP.

In [ ]:
# standards packages pour nettoyage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Afficher la barre des erreurs
import missingno as msno

#modélisation
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
import scipy.stats as stats

import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

---
## Phase 1 — Business Understanding

### Contexte

Vous êtes data analyst pour une société de revente de véhicules d'occasion.
La direction veut mieux comprendre **ce qui influence le prix de revente** (`selling_price`) des voitures.

### Objectif
Aider la plateforme à estimer le prix de vente d'un véhicule à partir de ses caractéristiques techniques et commerciales, afin d'améliorer la tarification automatique et la négociation.

### Questions métier à résoudre

1. L'année de fabrication (`year`) prédit-elle significativement le prix de revente ?
2. En combinant plusieurs variables (kilométrage, puissance, carburant, transmission...), peut-on bâtir un modèle prédictif fiable ?
3. Les variables catégorielles (type de carburant, transmission, propriétaire) ont-elles un impact mesurable sur le prix ?

### Traduction en problème ML

| Élément | Valeur |
|---------|--------|
| **Type de problème** | Régression (prédiction d'une valeur continue) |
| **Variable cible Y** | `selling_price` |
| **Variables candidates X** | `year`, `km_driven`, `max_power`, `engine`, `seats` + catégorielles (`fuel`, `transmission`, `owner`, `seller_type`) |
| **Métrique principale** | R² ajusté + RMSE |

---

> **A compléter :** Reformulez en une phrase la question métier que vous allez résoudre avec la régression linéaire.

**Votre réponse ici :**

_« ... »_

---
## Phase 2 — Data Understanding

### 2.1 Chargement des données

Chargez le fichier CSV téléchargé depuis Kaggle.
Adaptez le chemin selon votre environnement (Colab, local, etc.).

In [ ]:
# TODO : Chargez le CSV dans un DataFrame nommé df
# df = pd.read_csv("car_details_v3.csv")
# TODO : Affichez les 5 premières lignes


### 2.2 Exploration initiale

Commencez par comprendre la structure globale du dataset avant toute analyse.

In [ ]:
# TODO : Affichez les dimensions (shape)
# TODO : Affichez les types de colonnes (dtypes)
# TODO : Affichez le nombre de doublons
# TODO : Supprimez les doublons si nécessaire et vérifiez


### 2.3 Analyse des valeurs manquantes

In [ ]:
# TODO : Affichez le nombre et pourcentage de valeurs manquantes par colonne
# TODO : Visualisez les valeurs manquantes avec missingno (msno.bar ou msno.matrix)


### 2.4 Analyse des variables numériques

Analysez séparément chaque variable numérique : distribution, outliers, statistiques descriptives.

> **Note :** Les colonnes `mileage`, `engine`, `max_power` contiennent des unités textuelles.
> Extrayez d'abord les valeurs numériques pour pouvoir les analyser.

In [ ]:
# TODO : Pour chaque colonne ['mileage', 'engine', 'max_power'] :
#   Utilisez .str.extract(r'([\d.]+)') pour extraire la partie numérique
#   Convertissez en float


In [ ]:
# TODO : Affichez les statistiques descriptives des variables numériques uniquement
# TODO : Tracez un histogramme pour chaque variable numérique (utilisez df.hist())
# TODO : Tracez un boxplot pour chaque variable numérique pour détecter les outliers


### 2.5 Analyse des variables catégorielles

Analysez la distribution de chaque variable catégorielle avec `value_counts()` et des barplots.

In [ ]:
# TODO : Pour chaque colonne catégorielle ['fuel', 'transmission', 'seller_type', 'owner'] :
#   Affichez les valeurs distinctes value_counts()
#   Tracez un barplot de la distribution


### 2.6 Analyse de la variable cible — selling_price

Avant de modéliser, il faut comprendre la distribution de la variable cible.
La régression linéaire suppose que les résidus sont normalement distribués, ce qui est favorisé par une cible peu asymétrique.

In [ ]:
# TODO : Tracez l'histogramme de selling_price
# TODO : Calculez et affichez le skewness de selling_price df['selling'].skew()
# TODO : Concluez : la distribution est-elle compatible avec la régression linéaire ?


### 2.7 Transformation logarithmique de la cible

Pour corriger l'asymétrie, on applique la transformation `log1p` sur `selling_price`.
Cela permettra de linéariser les relations et de satisfaire les hypothèses de la régression.

> **Rappel :** `log1p(x) = log(1 + x)` — utilisé pour éviter log(0) si des prix valent 0.

In [ ]:
# TODO : Créez log_price = np.log1p(df['selling_price'])
# TODO : Calculez le skewness de log_price
# TODO : Comparez les deux distributions (2 histogrammes côte à côte)
# TODO : Concluez sur l'effet de la transformation


### 2.8 Relations features → log_price (H1 — Vérification de la linéarité)

**Hypothèse H1 :** Il doit exister une relation linéaire entre chaque feature X et la cible Y.

Visualisez les relations entre les variables explicatives et `log_price`.

In [ ]:
# TODO : Pour les variables numériques (year, km_driven, max_power, engine, mileage) :
#   Tracez un scatter plot de chaque variable vs log_price
# TODO : Pour les variables catégorielles (fuel, transmission, seller_type, owner) :
#   Tracez un boxplot : catégorie vs log_price
# TODO : Concluez : quelles variables semblent avoir une relation linéaire avec log_price ?


### 2.9 Matrice de corrélation

Analysez les corrélations entre les variables numériques et `log_price`.

In [ ]:
# TODO : Calculez et affichez la matrice de corrélation (heatmap seaborn)
# Incluez uniquement les colonnes numériques + log_price
# TODO : Identifiez les 3 variables les plus corrélées avec log_price


### 2.10 Observations et hypothèses

Résumez ce que vous avez appris sur les données. Ces observations doivent justifier les choix de la Phase 3.

**Vos observations ici :**

- Distribution de la cible : _..._
- Variables les plus corrélées avec log_price : _..._
- Outliers identifiés : _..._
- Variables catégorielles dominantes : _..._
- Hypothèse H1 (linéarité) : _..._

---
## Phase 3 — Data Preparation

### 3.1 Suppression des colonnes inutilisables

In [ ]:
# TODO : Supprimez les colonnes 'torque' (inutilisable) et 'name' (identifiant)
# TODO : Affichez les colonnes restantes


### 3.2 Gestion des valeurs manquantes

In [ ]:
# TODO : Supprimez les lignes contenant des valeurs manquantes (dropna)
# TODO : Vérifiez qu'il n'y a plus de NaN
# TODO : Affichez le shape avant/après


### 3.3 Traitement des outliers

Les outliers extrêmes identifiés en Phase 2 peuvent perturber le modèle linéaire.
On supprime les observations au-delà de 3 écarts-types pour les variables concernées.

In [ ]:
# TODO : Pour km_driven et selling_price :
#   Calculez la moyenne et l'écart-type
#   Supprimez les lignes où la valeur est à plus de 3 sigma de la moyenne

#TODO : Pour mealage et seats, gerez les max et min

# TODO : Affichez le shape avant/après


### 3.4 Encodage des variables catégorielles

Les algorithmes de régression linéaire ne traitent que des valeurs numériques.
On utilise `pd.get_dummies()` avec `drop_first=True` pour éviter la multicolinéarité.

In [ ]:
# TODO : Utilisez pd.get_dummies() pour encoder :
#   'fuel', 'transmission', 'seller_type', 'owner'
#   avec drop_first=True
# TODO : Affichez les nouvelles colonnes créées et le shape final


### 3.5 Vérification de la multicolinéarité (H5 — VIF)

**Hypothèse H5 :** Les variables explicatives ne doivent pas être fortement corrélées entre elles.

On utilise le **VIF (Variance Inflation Factor)** :
- VIF < 5 : acceptable
- VIF entre 5 et 10 : attention
- VIF > 10 : multicolinéarité problématique

In [ ]:
# TODO : Calculez le VIF pour chaque variable numérique
#   Utilisez variance_inflation_factor de statsmodels
#   (importé en Phase 0)
# TODO : Identifiez les variables avec VIF > 10
# TODO : Concluez sur la multicolinéarité


**Observation** : Quelle decision prendre?

### 3.6 Séparation features / cible et split train/test

In [ ]:
# TODO : Définissez X (toutes les colonnes sauf 'selling_price' et 'log_price')
# TODO : Définissez y = df['log_price']
# TODO : Effectuez un train_test_split avec test_size=0.2, random_state=42
# TODO : Affichez les dimensions de X_train, X_test, y_train, y_test


---
## Phase 4 — Modélisation

Nous allons construire 3 modèles progressifs :

| Modèle | Features | Objectif |
|--------|----------|----------|
| **Modèle 1** | `year` seul | Régression simple |
| **Modèle 2** | Numériques seules | Régression multiple |
| **Modèle 3** | Numériques + catégorielles | Modèle complet |

### 4.1 Modèle 1 — Régression simple : year -> log_price

In [ ]:
# TODO : Entraînez un LinearRegression sur X_train[['age']] -> y_train
# TODO : Affichez le coefficient beta_1 (slope) et l'intercept beta_0
# TODO : Interprétez : que signifie ce coefficient dans le contexte métier ?
# TODO : Tracez le nuage de points + droite de régression


### 4.2 Modèle 2 — Régression multiple : variables numériques

In [ ]:
# TODO : Sélectionnez les colonnes numériques continues :
#   ['age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats']
# TODO : Entraînez un LinearRegression sur ces features
# TODO : Affichez les coefficients sous forme de tableau trié


### 4.3 Modèle 3 — Modèle complet (numériques + catégorielles)

In [ ]:
# TODO : Entraînez un LinearRegression sur l'intégralité de X_train
# TODO : Affichez le nombre de features utilisées


---
## Phase 5 — Evaluation

### 5.1 Métriques de performance

Pour chaque modèle, calculez : MAE, RMSE, R², R² ajusté.

> **Rappel R² ajusté :**
> R2_adj = 1 - (1 - R2) x (n - 1) / (n - p - 1)
> Où n = nombre d'observations, p = nombre de features

In [ ]:
# TODO : Définissez une fonction evaluate_model(model, X_tr, X_te, y_tr, y_te, label) qui :
#   - Calcule les prédictions sur train ET test
#   - Calcule MAE, RMSE, R2, R2 ajusté pour train et test
#   - Affiche les résultats proprement
# TODO : Appelez cette fonction pour model1, model2, model3


### 5.2 H2 — Indépendance des erreurs

**Hypothèse H2 :** Les résidus ne doivent pas présenter de structure en fonction de l'ordre des observations (pas d'autocorrélation).

On trace les résidus en fonction de l'index des observations.

In [ ]:
# TODO : Calculez les résidus du modèle 3 sur le jeu de test
# TODO : Tracez résidus vs index des observations
# TODO : Concluez : observe-t-on une structure ou un pattern ?


**Conclusion H2 :**
_..._

### 5.3 H3 — Homoscédasticité (variance constante des résidus)

**Hypothèse H3 :** La variance des résidus doit être constante quelle que soit la valeur prédite.
Un nuage de points "en entonnoir" signale une hétéroscédasticité.

In [ ]:
# TODO : Tracez un scatter : valeurs prédites vs résidus
# TODO : Ajoutez une ligne horizontale à y=0
# TODO : Concluez : la variance des résidus est-elle constante ?


**Conclusion H3 :**
_..._

### 5.4 H4 — Normalité des résidus

**Hypothèse H4 :** Les résidus doivent suivre une distribution normale.
On vérifie avec un histogramme et un QQ-plot.

In [ ]:
# TODO : Tracez l'histogramme des résidus
# TODO : Tracez le QQ-plot (scipy.stats.probplot)
# TODO : Concluez : les résidus sont-ils approximativement normaux ?


**Conclusion H4 :**
_..._

### 5.5 Comparaison overfitting / underfitting

Comparez R2 train et R2 test pour chaque modèle.

In [ ]:
# TODO : Pour chaque modèle, calculez R2_train et R2_test
# TODO : Tracez un graphique en barres groupées
# TODO : Concluez : overfitting, underfitting, ou modèle équilibré ?


**Conclusion :**
_..._

---
## Phase 6 — Deployment / Interprétation

### 6.1 Interprétation métier des coefficients

Les coefficients représentent l'impact de chaque variable sur `log(selling_price)`.

> **Rappel :** Un coefficient beta signifie qu'une unité d'augmentation de X multiplie le prix par e^beta.

In [ ]:
# TODO : Créez un DataFrame avec le nom des features et leur coefficient (modèle 3)
# TODO : Triez par valeur absolue décroissante, gardez les 15 premiers
# TODO : Visualisez avec un barplot horizontal
# TODO : Coloriez en bleu les coefficients positifs, en rouge les négatifs


### 6.2 Réponses aux questions métier initiales

Reprenez les 3 questions posées en Phase 1 et répondez-y avec des chiffres à l'appui.

**Q1 — `year` prédit-il significativement `selling_price` ?**
> beta_1 = _..._ -> Interprétation : _..._
> R² simple = _..._ -> Conclusion : _..._

**Q2 — Le modèle multiple est-il plus performant ?**
> R2_ajusté (modèle 3) = _..._ -> Conclusion : _..._

**Q3 — Les variables catégorielles apportent-elles de la valeur ?**
> Comparez R2_ajusté modèle 2 vs modèle 3 : _..._
> Variable catégorielle la plus influente : _..._ (coefficient = _..._)

### 6.3 Recommandations et limites du modèle

**Résumé:**
- _..._

**Limites identifiées :**
- _..._

**Pistes d'amélioration :**
- _..._

**Recommandation finale à la direction :**
_Rédigez une recommandation concrète (2-3 phrases) basée sur vos résultats._